[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tragabytes/panel-viaje/blob/main/tests/fase1_wikidata_proximidad.ipynb)


# Fase 1 — Wikidata por proximidad geográfica como fuente de imagen para POIs

**Sesión 07 del proyecto Panel de Viaje**

## Contexto

La decisión 13 (sesión 06) establece que el POIModule usará una cascada en dos pasos:
1. **Inventario**: Overpass devuelve los POIs de cada pueblo (validado sesión 06).
2. **Enriquecimiento**: para cada POI seleccionado, intentar conseguir foto y texto:
   - Wikipedia REST por título construido (validado sesión 05).
   - **Wikidata por proximidad geográfica** ← lo que validamos en esta sesión.
   - Icono según tipo de POI como último recurso.

## Objetivo de esta sesión

Validar que una consulta SPARQL del tipo "dame items con P18 imagen en un radio de N metros alrededor de estas coordenadas" sirve para enriquecer con foto los POIs de Overpass que NO tienen artículo propio en Wikipedia en español.

El caso crítico es **Pedraza**: tiene 15 POIs en Overpass pero 0 artículos propios en Wikipedia según sesión 05. Si Wikidata por proximidad consigue fotos para estos POIs, la cascada cubre el 100%.

## Pruebas

- **P1** — Query base de proximidad. Probar 3 radios (50 m, 100 m, 200 m) sobre coordenadas conocidas.
- **P2** — Cruce real con el inventario Overpass de los 5 municipios de la muestra.
- **P3** — Calidad del match. Criterio de desempate cuando hay varios items en el radio.
- **P4** — Latencia y tamaño de respuesta.
- **P5** — Veredicto y resumen para la ficha de Wikidata SPARQL.

## Reglas del proyecto aplicables
- User-Agent identificativo obligatorio.
- Cache-buster en todas las URLs SPARQL (P07 sesión 05).
- Pausa ~1 s entre peticiones SPARQL.
- Ningún QID metido de memoria: todos se resuelven por consulta (P04 sesión 05).

---
## Celda 0 — Setup común

In [ ]:
import requests
import time
import math
import random
import json
from urllib.parse import quote

# Identificación del proyecto. Obligatorio para Wikidata y Overpass.
HEADERS = {
    "User-Agent": "PanelViaje/0.1 (https://github.com/tragabytes/panel-viaje; sesion07-wikidata-proximidad)"
}

SPARQL_ENDPOINT = "https://query.wikidata.org/sparql"
OVERPASS_ENDPOINT = "https://overpass-api.de/api/interpreter"

PAUSA_SPARQL = 1.2  # segundos entre peticiones SPARQL
PAUSA_OVERPASS = 5.0  # segundos entre peticiones Overpass (P09 sesión 06)

def cache_buster():
    """Devuelve un parámetro aleatorio para evitar cachés intermedias (P07)."""
    return f"&_cb={random.randint(1, 10**9)}"

def sparql_query(query, timeout=60):
    """Ejecuta una consulta SPARQL contra Wikidata y devuelve (resultado_json, latencia_ms, bytes)."""
    url = f"{SPARQL_ENDPOINT}?format=json&query={quote(query)}{cache_buster()}"
    t0 = time.time()
    r = requests.get(url, headers=HEADERS, timeout=timeout)
    latencia_ms = int((time.time() - t0) * 1000)
    tam = len(r.content)
    r.raise_for_status()
    return r.json(), latencia_ms, tam

def sparql_query_reintentos(query, timeout=60, max_intentos=3):
    """Versión de sparql_query con reintentos ante 502/503/504.
    Añadida en la iteración de la sesión 07 tras ver errores transitorios de Wikidata."""
    for intento in range(max_intentos):
        try:
            url = f"{SPARQL_ENDPOINT}?format=json&query={quote(query)}{cache_buster()}"
            t0 = time.time()
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            latencia_ms = int((time.time() - t0) * 1000)
            tam = len(r.content)
            if r.status_code in (502, 503, 504):
                espera = 2 * (intento + 1)
                print(f"    (HTTP {r.status_code}, reintento en {espera}s...)")
                time.sleep(espera)
                continue
            r.raise_for_status()
            return r.json(), latencia_ms, tam
        except requests.exceptions.RequestException as e:
            if intento == max_intentos - 1:
                raise
            espera = 2 * (intento + 1)
            print(f"    (error {type(e).__name__}, reintento en {espera}s...)")
            time.sleep(espera)
    raise RuntimeError("Se agotaron los reintentos SPARQL")

def haversine_m(lat1, lon1, lat2, lon2):
    """Distancia en metros entre dos puntos GPS."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return 2 * R * math.asin(math.sqrt(a))

print("Setup OK")
print(f"User-Agent: {HEADERS['User-Agent']}")

---
## Celda 1 — Resolver QIDs de los 5 municipios de la muestra

Regla del proyecto (P04 sesión 05): los QIDs **nunca** se meten de memoria. Se resuelven por consulta verificable al inicio de cada notebook. Resolvemos Medinaceli, Albarracín, Chinchón, Trujillo y Pedraza.

In [ ]:
MUESTRA_NOMBRES = [
    ("Medinaceli", "Soria"),
    ("Albarracín", "Teruel"),
    ("Chinchón", "Madrid"),
    ("Trujillo", "Cáceres"),
    ("Pedraza", "Segovia"),
]

def resolver_qid_municipio(nombre, provincia):
    """Resuelve el QID de un municipio español por nombre + provincia.
    
    Busca items que sean instancia directa de municipio de España (Q2074737)
    y cuyo label en español coincida exactamente con el nombre dado.
    El filtro por provincia es un filtro suave por el label de P131 (located in).
    """
    q = f'''
    SELECT ?item ?itemLabel ?provLabel ?lat ?lon WHERE {{
      ?item wdt:P31 wd:Q2074737 ;
            rdfs:label "{nombre}"@es .
      OPTIONAL {{ ?item wdt:P131 ?prov . ?prov rdfs:label ?provLabel . FILTER(LANG(?provLabel) = "es") }}
      OPTIONAL {{ ?item p:P625/psv:P625 ?coord .
                   ?coord wikibase:geoLatitude ?lat ;
                          wikibase:geoLongitude ?lon . }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "es". }}
    }}
    '''
    data, lat_ms, _ = sparql_query(q)
    # Filtrar por provincia en post-proceso
    candidatos = []
    for b in data["results"]["bindings"]:
        prov_label = b.get("provLabel", {}).get("value", "")
        qid = b["item"]["value"].split("/")[-1]
        lat = float(b["lat"]["value"]) if "lat" in b else None
        lon = float(b["lon"]["value"]) if "lon" in b else None
        candidatos.append({"qid": qid, "prov": prov_label, "lat": lat, "lon": lon})
    # Buscar match por provincia
    for c in candidatos:
        if provincia.lower() in c["prov"].lower():
            return c, lat_ms
    # Si no hay match por provincia, devolver el primero con coordenadas
    for c in candidatos:
        if c["lat"] is not None:
            return c, lat_ms
    return (candidatos[0] if candidatos else None), lat_ms

muestra = []
for nombre, prov in MUESTRA_NOMBRES:
    c, ms = resolver_qid_municipio(nombre, prov)
    if c is None:
        print(f"✗ {nombre}: NO RESUELTO")
    else:
        print(f"✓ {nombre:<14} → {c['qid']:<10} ({c['prov']})  [{c['lat']:.4f}, {c['lon']:.4f}]  ({ms} ms)")
        muestra.append({"nombre": nombre, "provincia": prov, **c})
    time.sleep(PAUSA_SPARQL)

print(f"\nMuestra resuelta: {len(muestra)}/{len(MUESTRA_NOMBRES)}")

---
## Celda 2 — Obtener el inventario de POIs de Overpass para los 5 municipios

Replicamos la query validada en sesión 06: POIs en un radio de 1500 m alrededor del centro de cada municipio, filtrados por tags `historic=*`, `tourism=viewpoint|attraction`, `natural=peak`, `building=castle|church|monastery|cathedral|chapel`.

Guardamos el resultado en `pois_por_municipio` para usarlo en las pruebas siguientes.

In [ ]:
def fetch_overpass(query, timeout_cliente=120):
    """Ejecuta una query Overpass. Usa los timeouts recomendados por P09 sesión 06."""
    t0 = time.time()
    r = requests.post(OVERPASS_ENDPOINT, data={"data": query}, headers=HEADERS, timeout=timeout_cliente)
    latencia_ms = int((time.time() - t0) * 1000)
    tam = len(r.content)
    r.raise_for_status()
    return r.json(), latencia_ms, tam

def query_pois_municipio(lat, lon, radio_m=1500):
    return f'''
[out:json][timeout:180];
(
  node(around:{radio_m},{lat},{lon})[historic];
  way(around:{radio_m},{lat},{lon})[historic];
  node(around:{radio_m},{lat},{lon})[tourism~"^(viewpoint|attraction)$"];
  way(around:{radio_m},{lat},{lon})[tourism~"^(viewpoint|attraction)$"];
  node(around:{radio_m},{lat},{lon})[natural=peak];
  node(around:{radio_m},{lat},{lon})[building~"^(castle|church|monastery|cathedral|chapel)$"];
  way(around:{radio_m},{lat},{lon})[building~"^(castle|church|monastery|cathedral|chapel)$"];
);
out center;
'''

def extraer_pois(data):
    """Extrae una lista limpia de POIs con nombre y coordenadas.
    Usa 'out center' para que los ways también tengan lat/lon (del centroide).
    """
    pois = []
    for el in data.get("elements", []):
        tags = el.get("tags", {})
        nombre = tags.get("name")
        if not nombre:
            continue
        # Coordenadas: nodes tienen lat/lon directo, ways tienen center
        if "lat" in el and "lon" in el:
            lat, lon = el["lat"], el["lon"]
        elif "center" in el:
            lat, lon = el["center"]["lat"], el["center"]["lon"]
        else:
            continue
        # Tipo principal para priorización
        tipo = tags.get("historic") or tags.get("tourism") or tags.get("building") or tags.get("natural") or "other"
        pois.append({
            "nombre": nombre,
            "lat": lat,
            "lon": lon,
            "tipo": tipo,
            "osm_id": el.get("id"),
            "osm_type": el.get("type"),
        })
    return pois

pois_por_municipio = {}
for m in muestra:
    print(f"→ {m['nombre']}...", end=" ")
    try:
        q = query_pois_municipio(m["lat"], m["lon"])
        data, ms, tam = fetch_overpass(q)
        pois = extraer_pois(data)
        pois_por_municipio[m["nombre"]] = pois
        print(f"{len(pois)} POIs con nombre ({ms} ms, {tam} bytes)")
    except Exception as e:
        print(f"ERROR: {e}")
        pois_por_municipio[m["nombre"]] = []
    time.sleep(PAUSA_OVERPASS)

print("\nResumen del inventario:")
for nombre, pois in pois_por_municipio.items():
    print(f"  {nombre:<14} {len(pois):>3} POIs")
    # Mostrar los 3 primeros como muestra
    for p in pois[:3]:
        print(f"                 · {p['nombre']} ({p['tipo']})")

---
## Prueba 1 — Query base de proximidad en 3 radios

Objetivo: validar que la consulta SPARQL de proximidad geográfica funciona y ver cuánto "ruido" trae cada radio.

Usamos las coordenadas del centro de cada municipio de la muestra como punto de prueba. Para cada uno, pedimos items Wikidata con P18 (imagen) en radios de 50, 100 y 200 metros. Contamos cuántos hay y miramos qué son.

**Nota técnica**: Wikidata expone el servicio geoespacial con `SERVICE wikibase:around`, que necesita un punto central construido con `wikibase:center` y un radio en km.

**Versión 2 de la query** (aplicada directamente aquí tras iterar en la sesión):
- `GROUP_CONCAT` sobre `?instanceLabel` para que un item con varios `wdt:P31` aparezca en una sola fila (el Palacio Ducal de Medinaceli es instancia de palacio, museo y monumento: antes salía 3 veces).
- `FILTER NOT EXISTS { ?item wdt:P31 wd:Q2074737 }` para excluir el propio municipio, que siempre aparecía a 0 m como primer resultado.
- Resultados ordenados por distancia ascendente para que el primer item sea el más cercano.
- Uso de `sparql_query_reintentos` para sobrevivir a errores 502/503/504 transitorios de Wikidata.

In [ ]:
def query_proximidad(lat, lon, radio_km):
    """Query SPARQL v2: una fila por QID (GROUP_CONCAT sobre instance)
    y excluye items instancia de "municipio de España" (Q2074737) para que
    el propio pueblo no contamine los resultados como match a 0 m.
    """
    return f'''
    SELECT ?item ?itemLabel ?itemDescription ?image ?coord
           (GROUP_CONCAT(DISTINCT ?instanceLabel; separator=" | ") AS ?instances)
    WHERE {{
      SERVICE wikibase:around {{
        ?item wdt:P625 ?coord .
        bd:serviceParam wikibase:center "Point({lon} {lat})"^^geo:wktLiteral .
        bd:serviceParam wikibase:radius "{radio_km}" .
      }}
      ?item wdt:P18 ?image .
      FILTER NOT EXISTS {{ ?item wdt:P31 wd:Q2074737 }}
      OPTIONAL {{
        ?item wdt:P31 ?instance .
        ?instance rdfs:label ?instanceLabel .
        FILTER(LANG(?instanceLabel) = "es")
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "es,en". }}
    }}
    GROUP BY ?item ?itemLabel ?itemDescription ?image ?coord
    LIMIT 50
    '''

def parsear_coord_wkt(coord_wkt):
    """Convierte 'Point(lon lat)' en (lat, lon)."""
    s = coord_wkt.replace("Point(", "").replace(")", "")
    lon_str, lat_str = s.split()
    return float(lat_str), float(lon_str)

def ejecutar_proximidad(lat, lon, radio_km):
    q = query_proximidad(lat, lon, radio_km)
    data, ms, tam = sparql_query_reintentos(q)
    items = []
    for b in data["results"]["bindings"]:
        qid = b["item"]["value"].split("/")[-1]
        label = b.get("itemLabel", {}).get("value", "")
        desc = b.get("itemDescription", {}).get("value", "")
        image = b.get("image", {}).get("value", "")
        instances = b.get("instances", {}).get("value", "")
        try:
            ilat, ilon = parsear_coord_wkt(b["coord"]["value"])
            dist = haversine_m(lat, lon, ilat, ilon)
        except Exception:
            ilat, ilon, dist = None, None, None
        items.append({
            "qid": qid, "label": label, "desc": desc, "image": image,
            "instance": instances, "lat": ilat, "lon": ilon, "dist_m": dist
        })
    # Ordenar por distancia para que el primero sea el más cercano
    items.sort(key=lambda x: x["dist_m"] if x["dist_m"] is not None else 1e9)
    return items, ms, tam

RADIOS_PRUEBA = [0.05, 0.1, 0.2]  # en km = 50 m, 100 m, 200 m
resultados_p1 = {}

for m in muestra:
    resultados_p1[m["nombre"]] = {}
    print(f"\n=== {m['nombre']} ({m['lat']:.4f}, {m['lon']:.4f}) ===")
    for r_km in RADIOS_PRUEBA:
        try:
            items, ms, tam = ejecutar_proximidad(m["lat"], m["lon"], r_km)
            resultados_p1[m["nombre"]][r_km] = {"items": items, "ms": ms, "tam": tam}
            print(f"  r={int(r_km*1000):>3} m  →  {len(items):>2} items  ({ms} ms, {tam} bytes)")
            for it in items[:5]:
                d_str = f"{it['dist_m']:.0f} m" if it["dist_m"] is not None else "?"
                inst = it["instance"][:30] if it["instance"] else ""
                print(f"      · {it['label']:<40} [{inst:<30}] {d_str}")
        except Exception as e:
            print(f"  r={int(r_km*1000):>3} m  →  ERROR: {e}")
            resultados_p1[m["nombre"]][r_km] = {"items": [], "ms": 0, "tam": 0}
        time.sleep(PAUSA_SPARQL)

---
## Prueba 2 — Cruce real con el inventario Overpass

Objetivo: para cada POI del inventario Overpass, lanzar la consulta de proximidad sobre **las coordenadas del propio POI** y ver si Wikidata tiene un item con imagen cerca.

Para no disparar el número de peticiones, limitamos a los primeros 8 POIs por municipio (ya son 40 peticiones en total, suficiente para el veredicto). Usamos radio de 100 m como compromiso razonable (la sesión 06 estimó este valor en la decisión 13).

**Caso crítico**: Pedraza. Si aquí Wikidata por proximidad devuelve imágenes para los POIs de Overpass, la cascada cubre el caso peor y la decisión 13 queda validada.

In [ ]:
RADIO_CRUCE_KM = 0.1  # 100 m
MAX_POIS_POR_MUNICIPIO = 8

resultados_p2 = {}

for m in muestra:
    nombre = m["nombre"]
    pois = pois_por_municipio.get(nombre, [])[:MAX_POIS_POR_MUNICIPIO]
    print(f"\n=== {nombre} — cruzando {len(pois)} POIs con Wikidata (r=100 m) ===")
    por_poi = []
    for poi in pois:
        try:
            items, ms, tam = ejecutar_proximidad(poi["lat"], poi["lon"], RADIO_CRUCE_KM)
            hit = len(items) > 0
            marca = "✓" if hit else "·"
            top_label = items[0]["label"] if hit else "—"
            top_dist = f"{items[0]['dist_m']:.0f} m" if hit and items[0]["dist_m"] is not None else ""
            print(f"  {marca} {poi['nombre']:<40} [{poi['tipo']:<12}] → {len(items)} match(es)  {top_label} {top_dist}")
            por_poi.append({"poi": poi, "items": items, "ms": ms})
        except Exception as e:
            print(f"  ✗ {poi['nombre']}: ERROR {e}")
            por_poi.append({"poi": poi, "items": [], "ms": 0})
        time.sleep(PAUSA_SPARQL)
    resultados_p2[nombre] = por_poi

# Resumen de cobertura
print("\n" + "="*60)
print("Cobertura de Wikidata por proximidad (100 m) sobre POIs Overpass")
print("="*60)
total_pois = 0
total_con_match = 0
for nombre, por_poi in resultados_p2.items():
    con_match = sum(1 for x in por_poi if len(x["items"]) > 0)
    total_pois += len(por_poi)
    total_con_match += con_match
    pct = (con_match / len(por_poi) * 100) if por_poi else 0
    print(f"  {nombre:<14} {con_match:>2}/{len(por_poi):<2}  ({pct:>5.1f}%)")
pct_total = (total_con_match / total_pois * 100) if total_pois else 0
print(f"  {'TOTAL':<14} {total_con_match:>2}/{total_pois:<2}  ({pct_total:>5.1f}%)")

---
## Prueba 3 — Calidad del match

Objetivo: cuando la query devuelve varios items dentro del radio, necesitamos un criterio para elegir el correcto. Un match "bueno" es aquel en el que el item Wikidata se corresponde realmente con el POI de Overpass, no con otra cosa cercana.

Definimos un score de match combinando:
- **Similitud de label** (comparación simple de palabras significativas entre el nombre OSM y el label Wikidata).
- **Distancia** (más cerca mejor).

Clasificamos cada POI con match en tres categorías:
- **Seguro**: label muy similar Y distancia < 30 m.
- **Probable**: label algo similar O distancia < 50 m.
- **Dudoso**: el resto.

In [ ]:
import unicodedata
import re

STOP_ES = {"de", "del", "la", "el", "los", "las", "y", "a", "en", "san", "santa", "santo",
           "nuestra", "señora", "virgen", "iglesia", "ermita", "castillo", "torre", "convento",
           "monasterio", "catedral", "capilla"}

def normaliza(s):
    s = s.lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return s.strip()

def palabras_significativas(s):
    return {w for w in normaliza(s).split() if w and w not in STOP_ES and len(w) > 2}

def similitud_label(a, b):
    """Jaccard sobre palabras significativas. 0 a 1."""
    pa, pb = palabras_significativas(a), palabras_significativas(b)
    if not pa or not pb:
        return 0.0
    inter = pa & pb
    union = pa | pb
    return len(inter) / len(union)

def clasificar_match(poi_nombre, item):
    sim = similitud_label(poi_nombre, item["label"])
    dist = item["dist_m"] if item["dist_m"] is not None else 999
    if sim >= 0.5 and dist < 30:
        return "seguro", sim, dist
    if sim >= 0.3 or dist < 50:
        return "probable", sim, dist
    return "dudoso", sim, dist

def mejor_item(poi_nombre, items):
    """Elige el mejor item por score combinado: similitud alta + distancia baja."""
    if not items:
        return None
    def score(it):
        sim = similitud_label(poi_nombre, it["label"])
        dist = it["dist_m"] if it["dist_m"] is not None else 999
        # Normalizamos distancia a 0-1 (100 m max considerado)
        dist_norm = 1 - min(dist, 100) / 100
        return sim * 0.6 + dist_norm * 0.4
    return max(items, key=score)

# Aplicar clasificación a los resultados de P2
conteo = {"seguro": 0, "probable": 0, "dudoso": 0, "sin_match": 0}
ejemplos = {"seguro": [], "probable": [], "dudoso": []}

for nombre, por_poi in resultados_p2.items():
    print(f"\n=== {nombre} ===")
    for entry in por_poi:
        poi = entry["poi"]
        items = entry["items"]
        if not items:
            conteo["sin_match"] += 1
            print(f"  ·  {poi['nombre']:<40}  (sin match)")
            continue
        best = mejor_item(poi["nombre"], items)
        clase, sim, dist = clasificar_match(poi["nombre"], best)
        conteo[clase] += 1
        marca = {"seguro": "★", "probable": "✓", "dudoso": "?"}[clase]
        print(f"  {marca}  {poi['nombre']:<40}  →  {best['label']:<35}  sim={sim:.2f}  d={dist:.0f}m  [{clase}]")
        if len(ejemplos[clase]) < 3:
            ejemplos[clase].append((poi["nombre"], best["label"], sim, dist))

print("\n" + "="*60)
print("Resumen de calidad del match")
print("="*60)
total = sum(conteo.values())
for k in ["seguro", "probable", "dudoso", "sin_match"]:
    pct = (conteo[k] / total * 100) if total else 0
    print(f"  {k:<12} {conteo[k]:>3}  ({pct:>5.1f}%)")

---
## Prueba 4 — Latencia y tamaño de respuesta

Objetivo: caracterizar el coste de la consulta de proximidad para poder decidir la frecuencia y estrategia de caché en fase 2.

Calculamos estadísticas sobre todas las consultas que ya hemos hecho en P1 y P2.

In [ ]:
latencias = []
tamanos = []

# De P1
for nombre, por_radio in resultados_p1.items():
    for r_km, info in por_radio.items():
        if info["ms"] > 0:
            latencias.append(info["ms"])
            tamanos.append(info["tam"])

# De P2 (solo tenemos ms, no tam — lo recogemos si se quiere más preciso en próxima iter)
for nombre, por_poi in resultados_p2.items():
    for entry in por_poi:
        if entry.get("ms", 0) > 0:
            latencias.append(entry["ms"])

def stats(xs):
    if not xs:
        return None
    xs_s = sorted(xs)
    n = len(xs_s)
    mediana = xs_s[n//2]
    media = sum(xs_s) / n
    return {"n": n, "min": xs_s[0], "max": xs_s[-1], "mediana": mediana, "media": int(media)}

s_lat = stats(latencias)
s_tam = stats(tamanos)

print("Latencia de la query de proximidad SPARQL")
print(f"  n={s_lat['n']}  min={s_lat['min']} ms  mediana={s_lat['mediana']} ms  media={s_lat['media']} ms  max={s_lat['max']} ms")

if s_tam:
    print("\nTamaño de respuesta (bytes, solo P1)")
    print(f"  n={s_tam['n']}  min={s_tam['min']}  mediana={s_tam['mediana']}  media={s_tam['media']}  max={s_tam['max']}")

# Estimación de coste por viaje
print("\nEstimación de coste en fase 2:")
print("  Supuesto: 4 pueblos cercanos activos, 2-3 POIs seleccionados por pueblo = ~10 consultas de proximidad.")
print("  Con caché agresiva, se ejecutan solo cuando cambia la lista de pueblos cercanos (~cada 15-30 min).")
if s_lat:
    coste_ms = 10 * s_lat["mediana"]
    print(f"  Coste estimado por refresco completo: ~{coste_ms} ms en consultas (secuenciales con pausa 1 s).")

---
## Prueba 5 — Veredicto y resumen para la ficha

Esta celda sintetiza los hallazgos en un formato que se puede copiar directamente al `seguimiento.json` del proyecto:
- Actualización de la ficha de Wikidata SPARQL (ampliación del caso de uso).
- Estado final de la decisión 13.
- Siguiente paso: fase 2.

In [ ]:
print("=" * 70)
print("RESUMEN SESIÓN 07 — Wikidata por proximidad geográfica")
print("=" * 70)

print("\n1) Muestra usada")
for m in muestra:
    n_pois = len(pois_por_municipio.get(m["nombre"], []))
    print(f"   · {m['nombre']:<14} QID={m['qid']:<10} POIs Overpass={n_pois}")

print("\n2) P1 — Query base de proximidad")
print("   Validamos 3 radios (50/100/200 m) sobre las coordenadas del centro de cada municipio.")
for nombre, por_radio in resultados_p1.items():
    counts = "  ".join(f"r={int(r*1000)}m→{len(info['items'])}" for r, info in por_radio.items())
    print(f"   · {nombre:<14} {counts}")

print("\n3) P2 — Cobertura sobre POIs reales de Overpass (r=100 m)")
total_pois = sum(len(x) for x in resultados_p2.values())
total_con_match = sum(1 for por_poi in resultados_p2.values() for x in por_poi if len(x["items"]) > 0)
print(f"   · Cobertura global: {total_con_match}/{total_pois} POIs con al menos un item Wikidata cercano")
print(f"   · Caso crítico Pedraza: {sum(1 for x in resultados_p2.get('Pedraza', []) if len(x['items']) > 0)}/{len(resultados_p2.get('Pedraza', []))}")

print("\n4) P3 — Calidad del match")
total_clasif = sum(conteo.values())
for k in ["seguro", "probable", "dudoso", "sin_match"]:
    pct = (conteo[k] / total_clasif * 100) if total_clasif else 0
    print(f"   · {k:<12} {conteo[k]:>3}  ({pct:>5.1f}%)")

print("\n5) P4 — Latencia")
if s_lat:
    print(f"   · mediana {s_lat['mediana']} ms, rango {s_lat['min']}-{s_lat['max']} ms sobre {s_lat['n']} consultas")

print("\n6) Veredicto preliminar")
# Umbral razonable: si los matches seguros + probables superan el 40% es aprovechable
tasa_util = ((conteo["seguro"] + conteo["probable"]) / total_clasif * 100) if total_clasif else 0
print(f"   · Tasa de matches útiles (seguros+probables): {tasa_util:.1f}%")
if tasa_util >= 40:
    print("   · VEREDICTO: APTA como fallback de imagen en la cascada de la decisión 13.")
    print("     - En fase 2, solo aceptar matches clasificados como 'seguro' o 'probable'.")
    print("     - Los 'dudoso' se descartan y el POI cae al icono por tipo.")
else:
    print("   · VEREDICTO: COBERTURA BAJA. Evaluar si vale la pena mantener el fallback")
    print("     o simplificar la cascada a Wikipedia + icono por tipo.")

print("\n7) Siguiente paso")
print("   Con esto la fase 1 queda al 100%. Arrancar fase 2 (módulos de datos).")